# Phase 3b — NER Extraction & Entity Features

**Input:** `data/contracts_ie_clean.csv` (produced by NB01)

**Output:** `data/ner_features.csv` — per-contract NER features for downstream classifiers

**Goal:** Extract named entities (ORG, GPE/LOC) from contract descriptions using spaCy,
then flag contracts where the same location appears across multiple contracts
(potential shell-company / address-sharing network).

---
Before running: `python -m spacy download en_core_web_sm`

## 0. Config

In [ ]:
import os

# Relative paths — no hardcoded user directories
DATA_DIR   = "../data"
INPUT_CSV  = os.path.join(DATA_DIR, "contracts_ie_clean.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "ner_features.csv")

# Set to None to process the full dataset; set to e.g. 1000 for a quick test run
SAMPLE_SIZE = None  # change to 1000 for a fast smoke-test

print(f"Input  : {os.path.abspath(INPUT_CSV)}")
print(f"Output : {os.path.abspath(OUTPUT_CSV)}")

## 1. Imports

In [ ]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

print(f"spaCy version: {spacy.__version__}")

## 2. Load Data

In [ ]:
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df):,} contracts")
print(f"Columns: {list(df.columns)}")

if SAMPLE_SIZE is not None:
    df = df.sample(SAMPLE_SIZE, random_state=42).copy()
    print(f"Using sample of {SAMPLE_SIZE:,} rows")
else:
    df = df.copy()
    print("Processing full dataset")

## 3. Load spaCy Model

In [ ]:
nlp = spacy.load("en_core_web_sm")
print(f"Model: {nlp.meta['name']} (lang={nlp.meta['lang']})")

## 4. Extract Named Entities

- **ORG** — organisations (potential bidders / shell companies)
- **GPE / LOC** — locations (for address-sharing detection)

**Column used:** `description` (cleaned text from NB01).  
> **Bug fix:** the original script used `cleaned_description` which does not exist in the dataset;  
> the correct column name is `description`.

In [ ]:
# Use 'description' — the cleaned text field produced by NB01
# (original script incorrectly used 'cleaned_description' — column does not exist)
texts = df["description"].fillna("").str.strip()
empty = texts.eq("")
texts[empty] = df.loc[empty, "title"].fillna("")

extracted_orgs = []
extracted_locs = []

for doc in tqdm(nlp.pipe(texts.astype(str), batch_size=64),
                total=len(texts), desc="NER"):
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    locs = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC")]
    extracted_orgs.append(orgs)
    extracted_locs.append(locs)

df["extracted_companies"] = extracted_orgs
df["extracted_locations"] = extracted_locs

print(f"\nExtraction complete.")
print(f"  Contracts with >=1 ORG : {sum(len(x)>0 for x in extracted_orgs):,}")
print(f"  Contracts with >=1 LOC : {sum(len(x)>0 for x in extracted_locs):,}")

## 5. Shared-Address Flag

A location appearing in more than one contract may indicate multiple bidders
sharing an address (potential shell-company network).
We flag any contract whose description mentions at least one such location.

In [ ]:
exploded_locs = df.explode("extracted_locations")
location_counts = exploded_locs["extracted_locations"].value_counts()

suspicious_locations = set(location_counts[location_counts > 1].index.tolist())
print(f"Unique locations found    : {len(location_counts):,}")
print(f"Suspicious locations (>1) : {len(suspicious_locations):,}")

df["shared_address_flag"] = df["extracted_locations"].apply(
    lambda locs: 1 if any(loc in suspicious_locations for loc in locs) else 0
)

flagged = int(df["shared_address_flag"].sum())
print(f"Contracts flagged : {flagged:,} ({flagged/len(df)*100:.1f}%)")

## 6. Save Output

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

ner_features = df[["contract_id", "shared_address_flag",
                    "extracted_companies", "extracted_locations"]]
ner_features.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(ner_features):,} rows -> {OUTPUT_CSV}")
ner_features.head()

## 7. EDA on NER Results

In [ ]:
# Top organisations mentioned across all contracts
all_orgs = [org for orgs in df["extracted_companies"] for org in orgs]
top_orgs = Counter(all_orgs).most_common(20)

fig, ax = plt.subplots(figsize=(10, 6))
orgs_df = pd.DataFrame(top_orgs, columns=["org", "count"])
ax.barh(orgs_df["org"][::-1], orgs_df["count"][::-1], color="steelblue")
ax.set_title("Top 20 organisations in contract descriptions (NER)")
ax.set_xlabel("Mentions")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "plot_top_ner_orgs.png"), dpi=150)
plt.show()

print("\nshared_address_flag distribution:")
print(df["shared_address_flag"].value_counts()
      .rename({0: "Clean", 1: "Flagged"}).to_string())